# 🚀 AI Media Studio — Google Colab GPU Worker v2.8 (Dual-World LivePortrait + LatentSync 1.5 / MuseTalk 1.5 + Vozes Regionais)

Worker remoto 1-clique otimizado para GPU **NVIDIA T4 (16GB VRAM `fp16`)** conectado via **Cloudflare Quick Tunnel (`trycloudflare.com`)** ao Painel VPS **Novela Studio v2.8**:
- **Stage 1 + Stage 2 Turn Animator (`POST /animate_turn`)**: LivePortrait (olhar, piscada, respiração, `lead_in_pause` e reação muda de vergonha) + LatentSync 1.5 / MuseTalk 1.5 (`fp16`)
- **Re-LipSync em Vídeos do Google Flow Veo 3.1 (`POST /lipsync_video`)**: Redublagem labial de clipes `.mp4` com áudio Gemini TTS PT-BR 48kHz via LatentSync 1.5 / MuseTalk 1.5
- **Compatibilidade v2.6/v2.7 (`POST /animate-take`)**, **Zero-Shot Voice Cloning F5-TTS-PT-BR (`POST /tts-clone`)** e **Seed-VC (`POST /voice-convert`)**

**Como usar:**
1. No Google Colab, ative **Runtime -> Change runtime type -> T4 GPU** (16GB VRAM).
2. Execute as células **1 -> 2 -> 3** em ordem.
3. Copie a URL `https://xxxx.trycloudflare.com` impressa na Célula 3 e cole na aba **Novela Studio -> Modo Colab GPU** no painel da VPS.

In [ ]:
# @title 1. Verificar GPU NVIDIA T4 e Instalar Dependências Base + Cloudflared
!nvidia-smi
!pip install -q fastapi uvicorn pydantic python-multipart requests huggingface_hub soundfile pydub opencv-python-headless scipy
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!cloudflared --version
print('✅ Ambiente base v2.8 (T4 fp16 + cloudflared) pronto!')

In [ ]:
# @title 2. Instalar Combo Open-Source v2.9 (LivePortrait + LatentSync 1.5 + MuseTalk 1.5 + Checkpoints Reais + Vozes)
ENGINE_CHOICE = 'liveportrait_latentsync'  # @param ['liveportrait_latentsync', 'liveportrait_musetalk', 'latentsync_1.5', 'musetalk_1.5']
ENABLE_STAGE1_LIVEPORTRAIT = True  # @param {type:'boolean'}
ENABLE_REGIONAL_VOICES = True  # @param {type:'boolean'}
HF_TOKEN = ''  # @param {type:'string'}
COLAB_AUTH_TOKEN = ''  # @param {type:'string'}

import os
from pathlib import Path
os.makedirs('/content/ai_lipsync', exist_ok=True)
os.makedirs('/content/ai_voices', exist_ok=True)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

!pip install -q huggingface_hub omegaconf pyyaml
!pip install -q tyro rich einops kornia imageio imageio-ffmpeg onnxruntime-gpu onnxruntime scikit-image albumentations lmdb diffusers transformers accelerate omegaconf safetensors insightface mediapipe face-alignment lpips librosa av pykalman ffmpeg-python python_speech_features scenedetect DeepCache moviepy gdown decord
from huggingface_hub import snapshot_download

if ENABLE_STAGE1_LIVEPORTRAIT:
    if not os.path.exists('/content/ai_lipsync/LivePortrait'):
        !git clone -q https://github.com/KwaiVGI/LivePortrait /content/ai_lipsync/LivePortrait || true
    if os.path.exists('/content/ai_lipsync/LivePortrait/requirements.txt'):
        !pip install -q -r /content/ai_lipsync/LivePortrait/requirements.txt || true
    print('📥 Baixando pesos reais KwaiVGI/LivePortrait...')
    snapshot_download('KwaiVGI/LivePortrait', local_dir='/content/ai_lipsync/LivePortrait/pretrained_weights', token=HF_TOKEN or None)

if not os.path.exists('/content/ai_lipsync/LatentSync'):
    !git clone -q https://github.com/bytedance/LatentSync /content/ai_lipsync/LatentSync || true
if os.path.exists('/content/ai_lipsync/LatentSync/requirements.txt'):
    !pip install -q -r /content/ai_lipsync/LatentSync/requirements.txt || true
print('📥 Baixando checkpoints ByteDance/LatentSync-1.5...')
snapshot_download('ByteDance/LatentSync-1.5', local_dir='/content/ai_lipsync/LatentSync/checkpoints', allow_patterns=['latentsync_unet.pt', 'whisper/*'], token=HF_TOKEN or None)

if not os.path.exists('/content/ai_lipsync/MuseTalk'):
    !git clone -q https://github.com/TMElyralab/MuseTalk /content/ai_lipsync/MuseTalk || true
if os.path.exists('/content/ai_lipsync/MuseTalk/requirements.txt'):
    !pip install -q -r /content/ai_lipsync/MuseTalk/requirements.txt || true
print('📥 Baixando checkpoints TMElyralab/MuseTalk...')
snapshot_download('TMElyralab/MuseTalk', local_dir='/content/ai_lipsync/MuseTalk/models', token=HF_TOKEN or None)

assert Path('/content/ai_lipsync/LatentSync/checkpoints/latentsync_unet.pt').is_file() or any(Path('/content/ai_lipsync/LatentSync/checkpoints').rglob('*.pt')), 'Checkpoints LatentSync-1.5 ausentes!'
assert Path('/content/ai_lipsync/MuseTalk/models').exists() and any(Path('/content/ai_lipsync/MuseTalk/models').rglob('*.pth')), 'Checkpoints MuseTalk ausentes!'

if ENABLE_REGIONAL_VOICES:
    !pip install -q f5-tts cached_path torchaudio || true
    if not os.path.exists('/content/ai_voices/seed-vc'):
        !git clone -q https://github.com/Plachtaa/seed-vc /content/ai_voices/seed-vc || true
    if os.path.exists('/content/ai_voices/seed-vc/requirements.txt'):
        !pip install -q -r /content/ai_voices/seed-vc/requirements.txt || true
    print('📥 Pré-baixando checkpoint F5-TTS-PT-BR (firstpixel/F5-TTS-pt-br)...')
    try:
        snapshot_download('firstpixel/F5-TTS-pt-br', local_dir='/content/ai_voices/f5_tts_pt_br', token=HF_TOKEN or None)
    except Exception as _hf_exc:
        print(f'Aviso F5-TTS-PT-BR HuggingFace: {_hf_exc}')

print(f'✅ Combo v2.9 com pesos reais T4 fp16 verificado: engine={ENGINE_CHOICE}, liveportrait={ENABLE_STAGE1_LIVEPORTRAIT}, voices={ENABLE_REGIONAL_VOICES}')

In [ ]:
# @title 3. Subir FastAPI Worker v2.9 (/health + /animate_turn + /lipsync_video + /animate-take + /tts-clone + /voice-convert)
import base64, json, os, re, shutil, subprocess, threading, time, uuid
from pathlib import Path
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn

LP_DIR = Path('/content/ai_lipsync/LivePortrait')
LS_DIR = Path('/content/ai_lipsync/LatentSync')
MT_DIR = Path('/content/ai_lipsync/MuseTalk')

def detect_gpu_info():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True
        ).strip().splitlines()[0]
        parts = [p.strip() for p in out.split(',')]
        name = parts[0] if parts else 'Colab GPU Runtime'
        mem_str = parts[1] if len(parts) > 1 else '15360 MiB'
        vram_gb = round(float(re.findall(r'[0-9.]+', mem_str)[0]) / 1024.0, 1) if re.findall(r'[0-9.]+', mem_str) else 15.0
        return name, vram_gb
    except Exception:
        return 'Colab GPU Runtime', 15.0

GPU_NAME, GPU_VRAM_GB = detect_gpu_info()
app = FastAPI(title='AI Media Studio Colab GPU Worker v2.9 (T4 fp16)')

class AnimateTurnRequest(BaseModel):
    image_b64: str
    audio_b64: str
    prompt: str = ''
    engine: str = 'liveportrait_latentsync'
    lipsync_engine: str = 'auto'
    emotion_preset: str = 'attack_angry'
    lead_in_pause: float = 0.0
    is_silent_reaction: bool = False
    width: int = 1080
    height: int = 1920

class LipsyncVideoRequest(BaseModel):
    video_b64: str
    audio_b64: str
    engine: str = 'latentsync_1.5'
    width: int = 1080
    height: int = 1920

class AnimateTakeRequest(BaseModel):
    image_b64: str
    audio_b64: str
    prompt: str = ''
    engine: str = 'liveportrait_latentsync'
    emotion_preset: str = 'warm'
    lead_in_pause: float = 0.0
    is_silent_reaction: bool = False
    width: int = 1080
    height: int = 1920

class TtsCloneRequest(BaseModel):
    text: str
    ref_audio_b64: str
    ref_text: str = ''
    character_id: str = 'dona_socorro'
    speed: float = 1.0

class VoiceConvertRequest(BaseModel):
    guide_audio_b64: str
    target_timbre_b64: str
    character_id: str = 'dona_socorro'
    diffusion_steps: int = 25
    semi_tone_shift: int = 0

def _probe_dur(path: Path) -> float:
    try:
        out = subprocess.check_output(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', str(path)],
            text=True,
        ).strip()
        return max(0.4, float(out or '2.0'))
    except Exception:
        return 2.0

def _normalize_mp4(raw_mp4: Path, wav_path: Path, out_mp4: Path, width: int, height: int) -> None:
    vid_dur = _probe_dur(raw_mp4)
    aud_dur = _probe_dur(wav_path)
    pad_tail = max(0.0, aud_dur - vid_dur + 0.12)
    vf = f'scale={width}:{height}:force_original_aspect_ratio=increase:flags=lanczos,crop={width}:{height},fps=30'
    if pad_tail > 0.04:
        vf += f',tpad=stop_mode=clone:stop_duration={pad_tail:.3f}'
    vf += ',format=yuv420p'
    subprocess.run([
        'ffmpeg', '-y', '-i', str(raw_mp4), '-i', str(wav_path),
        '-map', '0:v:0', '-map', '1:a:0',
        '-vf', vf, '-t', f'{aud_dur:.3f}',
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
        '-c:a', 'aac', '-b:a', '192k', '-ar', '48000', '-ac', '2', str(out_mp4),
    ], check=True)

def _ensure_weights_dir(repo_id: str, local_dir: Path, required_glob: str, allow_patterns=None) -> None:
    if not local_dir.exists() or not any(local_dir.rglob(required_glob)):
        from huggingface_hub import snapshot_download
        local_dir.mkdir(parents=True, exist_ok=True)
        dl_kw = {'local_dir': str(local_dir), 'token': os.environ.get('HF_TOKEN') or None}
        if allow_patterns:
            dl_kw['allow_patterns'] = allow_patterns
        snapshot_download(repo_id, **dl_kw)
    if not any(local_dir.rglob(required_glob)):
        raise RuntimeError(f'Checkpoints obrigatórios não encontrados em {local_dir} ({required_glob})')

def _stage1_liveportrait(img_path: Path, wav_path: Path, stage1_mp4: Path, emotion_preset: str, lead_in_pause: float, is_silent: bool) -> str:
    _ensure_weights_dir('KwaiVGI/LivePortrait', LP_DIR / 'pretrained_weights', '*.pth')
    if not (LP_DIR / 'inference.py').is_file():
        raise RuntimeError('LivePortrait inference.py não encontrado; execute a Célula 2 primeiro.')
    drive_preset = 'silent_shame_reaction' if is_silent else (emotion_preset or 'attack_angry')
    drv_dir = LP_DIR / 'assets/examples/driving'
    drv = (
        next(drv_dir.glob(f'*{drive_preset}*.pkl'), None)
        or next(drv_dir.glob('*.pkl'), None)
        or next(drv_dir.glob('d*.mp4'), None)
    )
    if not drv:
        raise RuntimeError(f'Template driving do LivePortrait ausente em {drv_dir}')
    lp_out = stage1_mp4.parent / 'lp_out'
    lp_out.mkdir(parents=True, exist_ok=True)
    cmd = [
        'python', str(LP_DIR / 'inference.py'),
        '-s', str(img_path),
        '-d', str(drv),
        '-o', str(lp_out),
        '--flag-use-half-precision',
    ]
    proc = subprocess.run(cmd, cwd=str(LP_DIR), capture_output=True, text=True)
    cand = next(iter(sorted(lp_out.glob('*.mp4'))), None)
    if proc.returncode != 0 or not cand or not cand.is_file():
        raise RuntimeError(f'LivePortrait fp16 falhou ({proc.returncode}): {proc.stderr[-400:]}')
    shutil.copy2(cand, stage1_mp4)
    return 'liveportrait_fp16'

def _stage2_lipsync_video(video_in: Path, wav_path: Path, out_mp4: Path, lipsync_engine: str, width: int, height: int) -> str:
    eng = (lipsync_engine or 'auto').lower()
    use_mt = 'musetalk' in eng
    errs = []
    if not use_mt:
        try:
            _ensure_weights_dir('ByteDance/LatentSync-1.5', LS_DIR / 'checkpoints', '*.pt', allow_patterns=['latentsync_unet.pt', 'whisper/*'])
            if (LS_DIR / 'scripts/inference.py').is_file():
                tmp_ls = out_mp4.with_name('ls_raw.mp4')
                cfg_rel = 'configs/unet/stage2.yaml'
                cmd = [
                    'python', '-m', 'scripts.inference',
                    '--unet_config_path', cfg_rel,
                    '--inference_ckpt_path', 'checkpoints/latentsync_unet.pt',
                    '--video_path', str(video_in),
                    '--audio_path', str(wav_path),
                    '--video_out_path', str(tmp_ls),
                    '--inference_steps', '20',
                    '--guidance_scale', '1.5',
                    '--enable_deepcache',
                ]
                proc = subprocess.run(cmd, cwd=str(LS_DIR), capture_output=True, text=True)
                if proc.returncode == 0 and tmp_ls.is_file():
                    _normalize_mp4(tmp_ls, wav_path, out_mp4, width, height)
                    return 'latentsync_1.5_fp16'
                errs.append(f'LatentSync rc={proc.returncode}: {proc.stderr[-250:]}')
        except Exception as exc:
            errs.append(f'LatentSync: {exc}')
    try:
        _ensure_weights_dir('TMElyralab/MuseTalk', MT_DIR / 'models', '*.pth')
        if (MT_DIR / 'scripts/inference.py').is_file():
            mt_out_dir = out_mp4.with_name('mt_out')
            mt_out_dir.mkdir(exist_ok=True)
            mt_cfg = out_mp4.with_name('mt_task.yaml')
            mt_cfg.write_text(f"task_0:\n  video_path: '{video_in}'\n  audio_path: '{wav_path}'\n", encoding='utf-8')
            cmd = [
                'python', '-m', 'scripts.inference',
                '--inference_config', str(mt_cfg),
                '--use_float16',
                '--result_dir', str(mt_out_dir),
            ]
            proc = subprocess.run(cmd, cwd=str(MT_DIR), capture_output=True, text=True)
            cand = next(iter(sorted(mt_out_dir.glob('**/*.mp4'))), None)
            if proc.returncode == 0 and cand and cand.is_file():
                _normalize_mp4(cand, wav_path, out_mp4, width, height)
                return 'musetalk_1.5_fp16'
            errs.append(f'MuseTalk rc={proc.returncode}: {proc.stderr[-250:]}')
    except Exception as exc:
        errs.append(f'MuseTalk: {exc}')
    raise RuntimeError('Inferência neural GPU (LatentSync 1.5 / MuseTalk 1.5) falhou: ' + ' | '.join(errs))

@app.get('/health')
def health(x_colab_token: str = Header(default='')):
    if COLAB_AUTH_TOKEN and x_colab_token != COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    return {
        'ok': True,
        'ready': True,
        'version': 'v2.9',
        'gpu': GPU_NAME,
        'vram_gb': GPU_VRAM_GB,
        'fp16': True,
        'engines': ['liveportrait', 'latentsync_1.5', 'musetalk_1.5', 'f5_tts_pt_br', 'seed_vc'],
        'routes': ['/health', '/animate_turn', '/lipsync_video', '/animate-take', '/tts-clone', '/voice-convert'],
        'active_engine': ENGINE_CHOICE,
    }

@app.post('/animate_turn')
def animate_turn(req: AnimateTurnRequest, x_colab_token: str = Header(default='')):
    if COLAB_AUTH_TOKEN and x_colab_token != COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    job_dir = Path(f'/tmp/colab_turn_{uuid.uuid4().hex[:8]}')
    job_dir.mkdir(parents=True, exist_ok=True)
    try:
        img_path = job_dir / 'portrait.jpg'
        wav_path = job_dir / 'speech.wav'
        stage1_dir = job_dir / 'stage1'
        stage1_dir.mkdir(exist_ok=True)
        stage1_mp4 = stage1_dir / 'stage1.mp4'
        out_mp4 = job_dir / 'animated_turn.mp4'
        img_path.write_bytes(base64.b64decode(req.image_b64))
        wav_path.write_bytes(base64.b64decode(req.audio_b64))
        st1_eng = _stage1_liveportrait(img_path, wav_path, stage1_mp4, req.emotion_preset, req.lead_in_pause, req.is_silent_reaction)
        if req.is_silent_reaction:
            _normalize_mp4(stage1_mp4, wav_path, out_mp4, req.width, req.height)
            used_engine = f'{st1_eng}+silent_shame_reaction'
        else:
            stage1_trimmed = stage1_dir / 'stage1_trimmed.mp4'
            _normalize_mp4(stage1_mp4, wav_path, stage1_trimmed, min(req.width, 512), min(req.height, 896))
            ls_choice = req.lipsync_engine if req.lipsync_engine != 'auto' else req.engine
            st2_eng = _stage2_lipsync_video(stage1_trimmed, wav_path, out_mp4, ls_choice, req.width, req.height)
            used_engine = f'{st1_eng}+{st2_eng}'
        return {
            'ok': True,
            'engine': used_engine,
            'gpu': GPU_NAME,
            'duration': round(_probe_dur(out_mp4), 2),
            'is_silent_reaction': req.is_silent_reaction,
            'video_b64': base64.b64encode(out_mp4.read_bytes()).decode('ascii'),
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)[:400])
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)

@app.post('/lipsync_video')
def lipsync_video(req: LipsyncVideoRequest, x_colab_token: str = Header(default='')):
    if COLAB_AUTH_TOKEN and x_colab_token != COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    job_dir = Path(f'/tmp/colab_lsvid_{uuid.uuid4().hex[:8]}')
    job_dir.mkdir(parents=True, exist_ok=True)
    try:
        vid_in = job_dir / 'flow_clip.mp4'
        wav_path = job_dir / 'speech.wav'
        out_mp4 = job_dir / 'redubbed.mp4'
        vid_in.write_bytes(base64.b64decode(req.video_b64))
        wav_path.write_bytes(base64.b64decode(req.audio_b64))
        used_engine = _stage2_lipsync_video(vid_in, wav_path, out_mp4, req.engine, req.width, req.height)
        return {
            'ok': True,
            'engine': used_engine,
            'gpu': GPU_NAME,
            'duration': round(_probe_dur(out_mp4), 2),
            'video_b64': base64.b64encode(out_mp4.read_bytes()).decode('ascii'),
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)[:400])
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)

@app.post('/animate-take')
def animate_take(req: AnimateTakeRequest, x_colab_token: str = Header(default='')):
    return animate_turn(
        AnimateTurnRequest(
            image_b64=req.image_b64,
            audio_b64=req.audio_b64,
            prompt=req.prompt,
            engine=req.engine,
            emotion_preset=req.emotion_preset,
            lead_in_pause=req.lead_in_pause,
            is_silent_reaction=req.is_silent_reaction,
            width=req.width,
            height=req.height,
        ),
        x_colab_token=x_colab_token,
    )

def _run_f5_tts_pt_br(text: str, ref_wav: Path, ref_text: str, out_wav: Path, speed: float = 1.0) -> str:
    f5_dir = Path('/content/ai_voices/f5_tts_pt_br')
    ckpt = next(f5_dir.glob('**/*.pt'), None) or next(f5_dir.glob('**/*.safetensors'), None) if f5_dir.exists() else None
    vocab = next(f5_dir.glob('**/vocab.txt'), None) if f5_dir.exists() else None
    try:
        from f5_tts.api import F5TTS
        tts_kw = {'ckpt_file': str(ckpt)} if ckpt else {}
        if vocab:
            tts_kw['vocab_file'] = str(vocab)
        model = F5TTS(**tts_kw)
        model.infer(ref_file=str(ref_wav), ref_text=ref_text or '', gen_text=text, file_wave=str(out_wav), speed=float(speed))
        if out_wav.is_file() and out_wav.stat().st_size > 1000:
            return 'F5-TTS-PT-BR'
    except Exception as _f5_exc:
        print(f'F5-TTS fallback aviso: {_f5_exc}')
    subprocess.run(['ffmpeg', '-y', '-i', str(ref_wav), '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(out_wav)], check=True)
    return 'f5_tts_fallback_copy'

def _run_seed_vc(guide_wav: Path, target_wav: Path, out_dir: Path, out_wav: Path, steps: int = 25, semi: int = 0) -> str:
    svc_dir = Path('/content/ai_voices/seed-vc')
    if (svc_dir / 'inference.py').is_file():
        cmd = [
            'python', str(svc_dir / 'inference.py'),
            '--source', str(guide_wav),
            '--target', str(target_wav),
            '--output', str(out_dir),
            '--diffusion-steps', str(int(steps)),
            '--semi-tone-shift', str(int(semi)),
            '--f0-condition', 'True',
        ]
        if subprocess.run(cmd, cwd=str(svc_dir)).returncode == 0:
            gen = next(iter(sorted(out_dir.glob('vc_*.wav'))), None)
            if gen and gen.is_file():
                shutil.copy2(gen, out_wav)
                return 'seed-vc'
    subprocess.run(['ffmpeg', '-y', '-i', str(guide_wav), '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(out_wav)], check=True)
    return 'seed_vc_fallback_normalized'

@app.post('/tts-clone')
def tts_clone(req: TtsCloneRequest, x_colab_token: str = Header(default='')):
    if COLAB_AUTH_TOKEN and x_colab_token != COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    job_dir = Path(f'/tmp/colab_tts_{uuid.uuid4().hex[:8]}')
    job_dir.mkdir(parents=True, exist_ok=True)
    try:
        ref_wav = job_dir / 'ref.wav'
        out_wav = job_dir / 'out.wav'
        ref_wav.write_bytes(base64.b64decode(req.ref_audio_b64))
        used_engine = _run_f5_tts_pt_br(req.text, ref_wav, req.ref_text, out_wav, req.speed)
        dur = _probe_dur(out_wav)
        return {'ok': True, 'engine': used_engine, 'gpu': GPU_NAME, 'character_id': req.character_id, 'duration': round(dur, 2), 'audio_b64': base64.b64encode(out_wav.read_bytes()).decode('ascii')}
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)

@app.post('/voice-convert')
def voice_convert(req: VoiceConvertRequest, x_colab_token: str = Header(default='')):
    if COLAB_AUTH_TOKEN and x_colab_token != COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    job_dir = Path(f'/tmp/colab_vc_{uuid.uuid4().hex[:8]}')
    job_dir.mkdir(parents=True, exist_ok=True)
    try:
        guide_wav = job_dir / 'guide.wav'
        target_wav = job_dir / 'target.wav'
        out_dir = job_dir / 'vc_out'
        out_dir.mkdir(exist_ok=True)
        out_wav = job_dir / 'converted.wav'
        guide_wav.write_bytes(base64.b64decode(req.guide_audio_b64))
        target_wav.write_bytes(base64.b64decode(req.target_timbre_b64))
        used_engine = _run_seed_vc(guide_wav, target_wav, out_dir, out_wav, req.diffusion_steps, req.semi_tone_shift)
        dur = _probe_dur(out_wav)
        return {'ok': True, 'engine': used_engine, 'gpu': GPU_NAME, 'character_id': req.character_id, 'duration': round(dur, 2), 'audio_b64': base64.b64encode(out_wav.read_bytes()).decode('ascii')}
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)

threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8787, log_level='warning'), daemon=True).start()
time.sleep(2)
tunnel_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8787'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('⏳ Aguardando URL do túnel Cloudflare v2.9...')
for line in tunnel_proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n============================================================')
        print('🚀 COLE ESTA URL NO PAINEL DA VPS (Novela Studio v2.9 -> Colab GPU):')
        print(m.group(0))
        print('============================================================\n')